# 02 Cleaning

Lanjutan dari notebook 01. Di sini setiap file wilayah dirapikan satu per satu
sampai formatnya sama, lalu digabung dan dibersihkan.

Format akhir yang dituju:
- satu baris = satu cabang dalam satu bulan
- `periode` ditulis `YYYY-MM`
- `kode_cabang` ditulis `KC001` sampai `KC030`
- semua kolom nilai berupa angka dalam Rp juta
- nama cabang, kab/kota, dan wilayah diambil dari master

Kalau semua langkah sudah jalan, kode di notebook ini dipindah ke `src/extract.py`
dan `src/transform.py`.

In [1]:
import pandas as pd

RAW = "../data/raw/"
pd.set_option("display.max_columns", None)

# Kolom nilai yang dipakai di semua file
KOLOM_NILAI = ["kredit_outstanding", "dpk_giro", "dpk_tabungan", "dpk_deposito",
               "kredit_bermasalah", "pendapatan_bunga", "beban_bunga",
               "pendapatan_nonbunga", "biaya_operasional", "laba_sebelum_pajak"]

# Kolom yang diambil dari setiap file wilayah
KOLOM_AKHIR = ["periode", "kode_cabang"] + KOLOM_NILAI + ["total_dpk"]

In [2]:
master = pd.read_excel(RAW + "master_cabang.xlsx")
master.head(3)

,kode_cabang,nama_cabang,kab_kota,wilayah,latitude,longitude
0,KC001,Surabaya,Kota Surabaya,Surabaya Raya,-7.2575,112.7521
1,KC002,Sidoarjo,Kab. Sidoarjo,Surabaya Raya,-7.4478,112.7183
2,KC003,Gresik,Kab. Gresik,Surabaya Raya,-7.1559,112.6555


## 1. Wilayah 1 - Surabaya Raya

Yang diubah: nama kolom dijadikan huruf kecil dengan garis bawah, dan periode dari tanggal menjadi `YYYY-MM`.

In [3]:
wil1 = pd.read_excel(RAW + "laporan_wil1_surabaya_raya.xlsx")

wil1.columns = wil1.columns.str.lower().str.replace(" ", "_")
wil1["periode"] = pd.to_datetime(wil1["periode"]).dt.strftime("%Y-%m")

wil1 = wil1[KOLOM_AKHIR]
wil1.head(3)

,periode,kode_cabang,kredit_outstanding,dpk_giro,dpk_tabungan,dpk_deposito,kredit_bermasalah,pendapatan_bunga,beban_bunga,pendapatan_nonbunga,biaya_operasional,laba_sebelum_pajak,total_dpk
0,2024-01,KC001,3276419.20,1405380.93,1256532.78,1475573.76,54625.56,30774.41,9234.50,3137.70,13389.02,11288.60,4137487.47
1,2024-02,KC001,3339218.42,1409633.65,1260335.08,1463437.04,61748.49,31364.26,9187.73,2845.64,14129.81,10892.36,4133405.77
2,2024-03,KC001,3380114.68,1420796.10,1270315.29,1478900.64,55276.07,31748.39,9277.92,3175.82,14199.21,11447.07,4170012.03


## 2. Wilayah 2 - Malang Raya

Semua kolom dibaca sebagai teks (`dtype=str`). Kalau tidak, kode `007` akan terbaca sebagai angka `7`.

Yang diubah:
- periode `Jan-2024` menjadi `2024-01`
- kode `007` menjadi `KC007`
- angka `2.019.533,54` menjadi `2019533.54` (titik dihapus, koma diganti titik)

In [4]:
wil2 = pd.read_csv(RAW + "laporan_wil2_malang_raya.csv", sep=";", dtype=str)

bulan_id = {"Jan": "01", "Feb": "02", "Mar": "03", "Apr": "04", "Mei": "05", "Jun": "06",
            "Jul": "07", "Agu": "08", "Sep": "09", "Okt": "10", "Nov": "11", "Des": "12"}

# "Jan-2024": 3 huruf pertama = bulan, mulai huruf ke-5 = tahun
wil2["periode"] = wil2["periode"].str[4:] + "-" + wil2["periode"].str[:3].map(bulan_id)
wil2["kode_cabang"] = "KC" + wil2["kode_cabang"].str.zfill(3)

for kol in KOLOM_NILAI + ["total_dpk"]:
    teks = wil2[kol].str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    wil2[kol] = pd.to_numeric(teks)

wil2 = wil2[KOLOM_AKHIR]
wil2.head(3)

,periode,kode_cabang,kredit_outstanding,dpk_giro,dpk_tabungan,dpk_deposito,kredit_bermasalah,pendapatan_bunga,beban_bunga,pendapatan_nonbunga,biaya_operasional,laba_sebelum_pajak,total_dpk
0,2024-01,KC007,2019533.54,731065.66,881785.47,732344.47,33891.04,20400.65,4797.24,2374.78,10788.36,7189.82,2345195.60
1,2024-02,KC007,2044079.44,739552.93,892022.51,742100.00,29256.05,20648.60,4858.57,2523.87,10483.28,7830.62,2373675.44
2,2024-03,KC007,2026171.01,740883.25,893627.09,745241.27,32106.25,20467.70,4875.44,2436.85,11267.25,6761.85,2379751.61


In [5]:
# Sel kosong tetap kosong (NaN), nanti ditangani setelah semua file digabung
wil2.isna().sum().sum()

np.int64(14)

## 3. Wilayah 3 - Mataraman

Dibaca dengan `header=4` karena ada judul di atas tabel. Nama kolomnya diganti
satu per satu supaya sama dengan file lain.

Yang diubah:
- bulan `01/2024` menjadi `2024-01`
- kode `13` menjadi `KC013`
- angka `Rp 1,115,601.88` menjadi `1115601.88` (`Rp ` dan koma dihapus)

In [6]:
wil3 = pd.read_excel(RAW + "Laporan Wilayah III Mataraman.xlsx", header=4)

wil3 = wil3.rename(columns={
    "Bulan": "periode", "Kode": "kode_cabang", "Nama Cabang": "nama_cabang",
    "Kredit (Rp Juta)": "kredit_outstanding", "Giro": "dpk_giro",
    "Tabungan": "dpk_tabungan", "Deposito": "dpk_deposito", "Total DPK": "total_dpk",
    "Kredit Bermasalah": "kredit_bermasalah", "Pend. Bunga": "pendapatan_bunga",
    "Beban Bunga": "beban_bunga", "Pend. Non Bunga": "pendapatan_nonbunga",
    "Biaya Operasional": "biaya_operasional", "Laba Sblm Pajak": "laba_sebelum_pajak"})

# "01/2024": 2 karakter pertama = bulan, mulai karakter ke-4 = tahun
wil3["periode"] = wil3["periode"].str[3:] + "-" + wil3["periode"].str[:2]
wil3["kode_cabang"] = "KC" + wil3["kode_cabang"].astype(str).str.zfill(3)

for kol in KOLOM_NILAI + ["total_dpk"]:
    teks = wil3[kol].str.replace("Rp ", "", regex=False).str.replace(",", "", regex=False)
    wil3[kol] = pd.to_numeric(teks)

wil3 = wil3[KOLOM_AKHIR]
wil3.head(3)

,periode,kode_cabang,kredit_outstanding,dpk_giro,dpk_tabungan,dpk_deposito,kredit_bermasalah,pendapatan_bunga,beban_bunga,pendapatan_nonbunga,biaya_operasional,laba_sebelum_pajak,total_dpk
0,2024-01,KC013,1115601.88,424801.23,541341.07,443632.46,39143.15,11556.82,2888.24,1312.21,6066.11,3914.68,1409774.76
1,2024-02,KC013,1133404.69,421761.06,537466.87,440297.75,41886.30,11741.24,2866.85,1364.20,6802.84,3435.75,1399525.68
2,2024-03,KC013,1152767.36,425523.92,542262.03,444773.70,37686.00,11941.83,2894.89,1481.98,6394.08,4134.83,1412559.65


Nama cabang yang tidak konsisten tidak perlu diperbaiki di sini, karena kolom nama
dibuang dan nanti diambil ulang dari master berdasarkan kode cabang.

## 4. Wilayah 4 - Tapal Kuda

Urutan kolom yang berbeda tidak masalah, karena kolom dipilih berdasarkan nama.
Cukup nama kolomnya yang diganti, lalu tanggal `2024-01-31` dipotong menjadi `2024-01`.

In [7]:
wil4 = pd.read_csv(RAW + "wil4_tapalkuda_rekap.csv")

wil4 = wil4.rename(columns={
    "tanggal": "periode", "outstanding_kredit": "kredit_outstanding",
    "dana_giro": "dpk_giro", "dana_tabungan": "dpk_tabungan",
    "dana_deposito": "dpk_deposito", "kredit_kol345": "kredit_bermasalah"})

wil4["periode"] = wil4["periode"].str[:7]

wil4 = wil4[KOLOM_AKHIR]
wil4.head(3)

,periode,kode_cabang,kredit_outstanding,dpk_giro,dpk_tabungan,dpk_deposito,kredit_bermasalah,pendapatan_bunga,beban_bunga,pendapatan_nonbunga,biaya_operasional,laba_sebelum_pajak,total_dpk
0,2024-01,KC021,1421407.72,461242.01,558595.99,718281.86,50743.46,13397.37,4181.22,1568.72,7119.73,3665.15,1738119.86
1,2024-02,KC021,NaN,471254.01,570721.21,734248.30,44075.31,13461.12,4273.67,1417.45,6874.19,3730.71,1776223.52
2,2024-03,KC021,1438663.62,468165.96,566981.38,725671.00,47314.78,13560.02,4228.71,NaN,6994.06,3902.28,1760818.34


## 5. Wilayah 5 - Madura & Pantura

Kedua sheet digabung dulu. Kolom `periode` dibaca sebagai teks supaya `2024-01`
tidak diubah otomatis menjadi tanggal.

In [8]:
sheets = pd.read_excel(RAW + "Wil5_Madura_Pantura.xlsx", sheet_name=None, dtype={"periode": str})
wil5 = pd.concat(sheets.values(), ignore_index=True)

wil5 = wil5.rename(columns={
    "kd_cab": "kode_cabang", "cabang": "nama_cabang", "os_kredit": "kredit_outstanding",
    "giro": "dpk_giro", "tabungan": "dpk_tabungan", "deposito": "dpk_deposito",
    "npl_nominal": "kredit_bermasalah", "pdpt_bunga": "pendapatan_bunga",
    "pdpt_non_bunga": "pendapatan_nonbunga", "biaya_ops": "biaya_operasional",
    "laba_sbl_pajak": "laba_sebelum_pajak", "tot_dpk": "total_dpk"})

wil5["periode"] = wil5["periode"].str[:7]
wil5["kode_cabang"] = wil5["kode_cabang"].str.replace("-", "", regex=False)

wil5 = wil5[KOLOM_AKHIR]
wil5.head(3)

,periode,kode_cabang,kredit_outstanding,dpk_giro,dpk_tabungan,dpk_deposito,kredit_bermasalah,pendapatan_bunga,beban_bunga,pendapatan_nonbunga,biaya_operasional,laba_sebelum_pajak,total_dpk
0,2024-01,KC025,586542.36,179289.91,247875.69,271753.61,18345.10,5734.73,1612.25,724.37,3116.24,1730.61,698919.21
1,2024-01,KC026,464815.26,152432.50,179692.94,212907.03,29851.54,4395.15,1268.42,435.31,2969.62,592.42,545032.47
2,2024-01,KC027,659600.10,185966.27,247075.34,263902.23,20206.39,6147.77,1584.73,516.36,3294.91,1784.49,696943.84


## 6. Penggabungan

Kelima file sekarang punya kolom yang sama, jadi bisa ditumpuk menjadi satu tabel.

In [9]:
df = pd.concat([wil1, wil2, wil3, wil4, wil5], ignore_index=True)
print(df.shape)
df.dtypes

(727, 13)


periode                    str
kode_cabang                str
kredit_outstanding     float64
dpk_giro               float64
dpk_tabungan           float64
dpk_deposito           float64
kredit_bermasalah      float64
pendapatan_bunga       float64
beban_bunga            float64
pendapatan_nonbunga    float64
biaya_operasional      float64
laba_sebelum_pajak     float64
total_dpk              float64
dtype: object

Seharusnya 30 cabang x 24 bulan = 720 baris. Hasilnya 727 karena masih ada
8 baris duplikat dan 1 bulan yang hilang.

## 7. Hapus Duplikat

In [10]:
print("Sebelum:", len(df))
df = df.drop_duplicates().reset_index(drop=True)
print("Sesudah:", len(df))

# Dicek juga apakah masih ada cabang-bulan yang muncul dua kali dengan nilai berbeda
print("Kunci ganda:", df.duplicated(subset=["kode_cabang", "periode"]).sum())

Sebelum: 727
Sesudah: 719
Kunci ganda: 0


## 8. Perbaiki Satuan

Aturan yang dipakai: kalau suatu nilai lebih dari 1000 kali median cabangnya,
nilai itu dianggap ditulis dalam Rupiah penuh lalu dibagi 1.000.000.

Batas 1000 kali dipilih karena kenaikan wajar dari bulan ke bulan tidak mungkin
sebesar itu, sedangkan salah satuan membuat nilai sejuta kali lebih besar.

In [11]:
for kol in KOLOM_NILAI:
    median = df.groupby("kode_cabang")[kol].transform("median")
    salah = df[kol] > median * 1000

    if salah.sum() > 0:
        print(kol)
        print(df.loc[salah, ["periode", "kode_cabang", kol]])

    df.loc[salah, kol] = df.loc[salah, kol] / 1_000_000

kredit_outstanding
     periode kode_cabang  kredit_outstanding
536  2024-09       KC023        6.128630e+11


In [12]:
# Dicek ulang, kredit Bondowoso September 2024 sudah sesuai bulan sekitarnya
df[(df["kode_cabang"] == "KC023") & (df["periode"].between("2024-08", "2024-10"))][["periode", "kredit_outstanding"]]

,periode,kredit_outstanding
535,2024-08,611194.16
536,2024-09,612863.03
537,2024-10,617856.99


## 9. Cek Total DPK

Total DPK dihitung ulang dari giro + tabungan + deposito, lalu dibandingkan dengan kolom
`total_dpk` dari file. Selisih di bawah 1 (Rp 1 juta) diabaikan karena bisa terjadi
akibat pembulatan.

In [13]:
komponen = ["dpk_giro", "dpk_tabungan", "dpk_deposito"]
hitung_ulang = df[komponen].sum(axis=1)
lengkap = df[komponen + ["total_dpk"]].notna().all(axis=1)
beda = lengkap & ((df["total_dpk"] - hitung_ulang).abs() > 1)

df[beda][["periode", "kode_cabang", "total_dpk"]].assign(hitung_ulang=hitung_ulang)

,periode,kode_cabang,total_dpk,hitung_ulang
31,2024-08,KC002,2363777.45,2148888.59
64,2025-05,KC003,2019331.37,1835755.79
446,2025-03,KC019,1162513.98,1167513.98


Ada 3 baris yang totalnya tidak cocok. Komponennya yang dianggap benar, jadi kolom
`total_dpk` dari file tidak dipakai lagi setelah langkah ini.

Kolom ini masih berguna untuk satu hal: kalau giro, tabungan, atau deposito kosong
salah satunya, nilainya bisa dihitung dari total dikurangi dua komponen lainnya.

In [14]:
satu_kosong = df[komponen].isna().sum(axis=1) == 1

for kol in komponen:
    lain = [k for k in komponen if k != kol]
    isi = satu_kosong & df[kol].isna() & df["total_dpk"].notna()

    if isi.sum() > 0:
        print(kol, "diisi pada", isi.sum(), "baris")

    df.loc[isi, kol] = df.loc[isi, "total_dpk"] - df.loc[isi, lain].sum(axis=1)

dpk_giro diisi pada 2 baris
dpk_tabungan diisi pada 2 baris
dpk_deposito diisi pada 4 baris


## 10. Isi Sel Kosong dengan Rumus Laba

Laba sebelum pajak = pendapatan bunga - beban bunga + pendapatan non bunga - biaya operasional.

Kalau dari lima kolom ini hanya satu yang kosong, nilainya bisa dihitung dari empat
kolom lain. Hasilnya bukan perkiraan, karena memang mengikuti rumus.

In [15]:
pb = df["pendapatan_bunga"]
bb = df["beban_bunga"]
nb = df["pendapatan_nonbunga"]
bo = df["biaya_operasional"]
lb = df["laba_sebelum_pajak"]

rumus = {
    "pendapatan_bunga": lb + bb - nb + bo,
    "beban_bunga": pb + nb - bo - lb,
    "pendapatan_nonbunga": lb - pb + bb + bo,
    "biaya_operasional": pb - bb + nb - lb,
    "laba_sebelum_pajak": pb - bb + nb - bo,
}

kolom_laba = list(rumus.keys())
satu_kosong = df[kolom_laba].isna().sum(axis=1) == 1

for kol, nilai in rumus.items():
    isi = satu_kosong & df[kol].isna()
    if isi.sum() > 0:
        print(kol, "diisi pada", isi.sum(), "baris")
    df.loc[isi, kol] = nilai[isi]

pendapatan_bunga diisi pada 1 baris
beban_bunga diisi pada 6 baris
pendapatan_nonbunga diisi pada 7 baris
biaya_operasional diisi pada 1 baris
laba_sebelum_pajak diisi pada 3 baris


In [16]:
# Sisa sel kosong setelah dua langkah di atas
df[KOLOM_NILAI].isna().sum()

kredit_outstanding     7
dpk_giro               0
dpk_tabungan           0
dpk_deposito           0
kredit_bermasalah      1
pendapatan_bunga       0
beban_bunga            0
pendapatan_nonbunga    0
biaya_operasional      0
laba_sebelum_pajak     0
dtype: int64

Yang tersisa hanya kredit outstanding dan kredit bermasalah. Dua kolom ini tidak
punya rumus, jadi diisi dengan interpolasi di langkah 12.

## 11. Lengkapi Bulan yang Hilang

Dibuat kerangka berisi semua kombinasi 30 cabang x 24 bulan. Data digabungkan ke
kerangka ini, jadi bulan yang tidak ada akan muncul sebagai baris kosong.

In [17]:
semua_periode = sorted(df["periode"].unique())
kerangka = pd.MultiIndex.from_product([master["kode_cabang"], semua_periode],
                                      names=["kode_cabang", "periode"])

df = df.set_index(["kode_cabang", "periode"]).reindex(kerangka).reset_index()

baris_kosong = df[KOLOM_NILAI].isna().all(axis=1)
print("Jumlah baris:", len(df))
df[baris_kosong][["kode_cabang", "periode"]]

Jumlah baris: 720


,kode_cabang,periode
569,KC024,2025-06


Situbondo Juni 2025 ditambahkan. Kolom `is_estimasi` dibuat untuk menandai baris
yang nilainya bukan dari laporan asli.

In [18]:
df["is_estimasi"] = baris_kosong

## 12. Interpolasi

Sel yang masih kosong diisi dengan nilai tengah antara bulan sebelum dan sesudahnya
pada cabang yang sama. Cara ini dipilih karena saldo kredit dan dana biasanya
berubah sedikit demi sedikit dari bulan ke bulan.

Baris yang diisi dengan cara ini ditandai `is_estimasi = True`.

In [19]:
df = df.sort_values(["kode_cabang", "periode"]).reset_index(drop=True)

for kol in KOLOM_NILAI:
    kosong = df[kol].isna()
    df[kol] = df.groupby("kode_cabang")[kol].transform(lambda s: s.interpolate(limit_direction="both"))
    df.loc[kosong, "is_estimasi"] = True

print("Sel kosong tersisa:", df[KOLOM_NILAI].isna().sum().sum())
print("Baris estimasi:", df["is_estimasi"].sum())

Sel kosong tersisa: 0
Baris estimasi: 9


## 13. Rapikan Hasil Akhir

`total_dpk` dibuang, angka dibulatkan 2 desimal, lalu nama cabang, kab/kota, dan
wilayah diambil dari master.

In [20]:
df = df.drop(columns="total_dpk")
df[KOLOM_NILAI] = df[KOLOM_NILAI].round(2)

df = df.merge(master[["kode_cabang", "nama_cabang", "kab_kota", "wilayah"]], on="kode_cabang", how="left")

kinerja = df[["periode", "kode_cabang", "nama_cabang", "kab_kota", "wilayah"] + KOLOM_NILAI + ["is_estimasi"]]
print(kinerja.shape)
kinerja.head()

(720, 16)


,periode,kode_cabang,nama_cabang,kab_kota,wilayah,kredit_outstanding,dpk_giro,dpk_tabungan,dpk_deposito,kredit_bermasalah,pendapatan_bunga,beban_bunga,pendapatan_nonbunga,biaya_operasional,laba_sebelum_pajak,is_estimasi
0,2024-01,KC001,Surabaya,Kota Surabaya,Surabaya Raya,3276419.20,1405380.93,1256532.78,1475573.76,54625.56,30774.41,9234.50,3137.70,13389.02,11288.60,False
1,2024-02,KC001,Surabaya,Kota Surabaya,Surabaya Raya,3339218.42,1409633.65,1260335.08,1463437.04,61748.49,31364.26,9187.73,2845.64,14129.81,10892.36,False
2,2024-03,KC001,Surabaya,Kota Surabaya,Surabaya Raya,3380114.68,1420796.10,1270315.29,1478900.64,55276.07,31748.39,9277.92,3175.82,14199.21,11447.07,False
3,2024-04,KC001,Surabaya,Kota Surabaya,Surabaya Raya,3414783.39,1302644.72,1383637.08,1496563.18,59274.44,32287.45,9597.05,3425.81,14492.57,11623.64,False
4,2024-05,KC001,Surabaya,Kota Surabaya,Surabaya Raya,3461083.90,1304430.86,1282902.11,1488278.01,67290.22,32725.23,9493.12,3136.03,15033.54,11334.60,False


In [21]:
# Cek akhir: tiap cabang harus punya 24 bulan dan tidak ada sel kosong
print(kinerja.groupby("kode_cabang").size().unique())
print(kinerja.isna().sum().sum())

[24]
0


## 14. Target

Setiap sheet target diubah dari format lebar ke format panjang dengan `melt`,
lalu ketiga sheet digabung berdasarkan kode cabang dan periode.

In [22]:
target_sheets = pd.read_excel(RAW + "target_anggaran_2024_2025.xlsx", sheet_name=None)

kredit = target_sheets["Target Kredit"].drop(columns="nama_cabang").melt(
    id_vars="kode_cabang", var_name="periode", value_name="target_kredit")
dpk = target_sheets["Target DPK"].drop(columns="nama_cabang").melt(
    id_vars="kode_cabang", var_name="periode", value_name="target_dpk")
laba = target_sheets["Target Laba"].drop(columns="nama_cabang").melt(
    id_vars="kode_cabang", var_name="periode", value_name="target_laba")

target = kredit.merge(dpk, on=["kode_cabang", "periode"]).merge(laba, on=["kode_cabang", "periode"])
target = target.sort_values(["kode_cabang", "periode"]).reset_index(drop=True)

print(target.shape)
target.head()

(720, 5)


,kode_cabang,periode,target_kredit,target_dpk,target_laba
0,KC001,2024-01,3307000,4015000,10990
1,KC001,2024-02,3334000,4045000,11080
2,KC001,2024-03,3361000,4075000,11170
3,KC001,2024-04,3389000,4105000,11260
4,KC001,2024-05,3416000,4135000,11350


## 15. Cek dengan Data Referensi

Hasil cleaning dibandingkan dengan `data/reference/data_bersih_referensi.csv`,
yaitu data asli sebelum dibuat berantakan.

Untuk baris yang bukan estimasi, selisihnya harus nol (paling besar 0,05 karena
pembulatan). Baris estimasi memang tidak akan sama persis.

In [23]:
ref = pd.read_csv("../data/reference/data_bersih_referensi.csv")
cek = kinerja.merge(ref, on=["periode", "kode_cabang"], suffixes=("", "_ref"))

hasil = []
for kol in KOLOM_NILAI:
    selisih = (cek[kol] - cek[kol + "_ref"]).abs()
    hasil.append({
        "kolom": kol,
        "selisih_maks_asli": selisih[~cek["is_estimasi"]].max(),
        "selisih_maks_estimasi": selisih[cek["is_estimasi"]].max(),
    })

pd.DataFrame(hasil).round(2)

,kolom,selisih_maks_asli,selisih_maks_estimasi
0,kredit_outstanding,0.00,20710.99
1,dpk_giro,0.00,249.05
2,dpk_tabungan,0.00,350.79
3,dpk_deposito,0.00,915.89
4,kredit_bermasalah,0.00,440.10
5,pendapatan_bunga,0.00,46.53
6,beban_bunga,0.01,20.63
7,pendapatan_nonbunga,0.01,12.65
8,biaya_operasional,0.00,146.61
9,laba_sebelum_pajak,0.01,108.05


Semua baris asli sudah sama dengan referensi. Selisih hanya ada di 9 baris
estimasi hasil interpolasi, dan nilainya kecil dibanding saldo cabang.

## Ringkasan

| Langkah | Hasil |
|---|---|
| Duplikat | 8 baris dihapus |
| Salah satuan | 1 nilai dibagi 1.000.000 |
| Total DPK tidak cocok | 3 baris, total dihitung ulang dari komponen |
| Sel kosong diisi dari total DPK | 8 sel |
| Sel kosong diisi dari rumus laba | 18 sel |
| Sel kosong diisi interpolasi | 8 sel (7 kredit outstanding, 1 kredit bermasalah) |
| Bulan hilang ditambahkan | 1 baris |

Hasil akhir: 720 baris realisasi dan 720 baris target, tanpa sel kosong.

Versi final kode ini ada di `src/extract.py` dan `src/transform.py`. Untuk membuat
file di `data/processed/`, jalankan dari folder utama proyek:

```
python src/transform.py
```